# Propagation of Hole waveguide (Antiguide)

Empty hole in glass. N0 configuration in Bird


hi kirbee ❤️ 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from slab import SlabExact, plotlogf
from scipy.optimize import newton
from scipy.integrate import quad
from IPython.display import Video, HTML

import json

plt.style.use('dark_background')
%matplotlib widget

In [ ]:
A = SlabExact(scale=1e-6, Ts=[2., 2., 2.], xrefs=[300, 500, 300],
              ns=[1.44, 1, 1.44], wl=1e-6, symmetric=True)


In [ ]:
w = .5
s = 0
amp = 2.5

Lx = -w + s
Rx = w + s

def quadratic_piecewise(x):
    conds = [(x<Lx), ((Lx<=x)*(x<=Rx)), (x>Rx)]
    funcs = [0, lambda x: amp * (1 - ((x-s)/w)**2), 0]
    return np.piecewise(x, conds, funcs)

def uneven_cubic(x):
    conds = [(x<-.5), ((-.5<=x)*(x<=.5)), (x>.5)]
    funcs = [0, lambda x: 5*(x+.1)*(.25-x**2), 0]
    return np.piecewise(x, conds, funcs)
    
def modified_dist(x):
    return (x-1/4) * np.exp(-(x-1/2)**2)  

def odd_modified_dist(x):
    return x * np.exp(-x**2)
    
def normal_dist(x):
    return np.exp(-x**2)

In [ ]:
# Set input function
f0 = quadratic_piecewise
f0_shifted = lambda x: f0(x) + 1
# Set input field type
field_type = 'TE'

In [ ]:
A.plot_refractive_index()
A.add_1d_plot(f0_shifted)

# Check for guided modes

Doesn't seem like it should have any

In [ ]:
A.determinant_plot(-6, 6, -3, 7);

No guided modes.

## Leaky Modes

In [ ]:
leaky_bdry = A.Zi_from_Beta(A.K0 * A.n_low)

In [ ]:
A.determinant_plot(0, 15, -5, 5);
plt.vlines(leaky_bdry.real, *plt.gca().get_ylim(), color='lightgray', lw=1, ls="--");

In [ ]:
Z0 = newton(A.determinant, np.array(1.01*A.K_low+.05j))
print(Z0)
A.plot_points(Z0)

In [ ]:
FsZ = A.fields(Z0)

In [ ]:
A.plot_field_1d(FsZ, xs=np.linspace(-10, 10, 600), linewidth=2.5, label='leaky mode');

In [ ]:
A.plot_field_2d(FsZ, xs=np.linspace(-10, 10, 600), zmin=0, zmax=10, zref=500, colorbar=False, figsize=(10,5))
# plt.axis('equal');

# Propagation of input



In [ ]:
Propagator_Class = A.propagator()
method = 'eigvec'
Normalizer = A.normalizer(method)

In [ ]:
contour = A.real_contour(0.012, 6.2, 200)

P1e = Propagator_Class(contour, f0=f0, Lx=Lx, Rx=Rx,
                       sign=1, Normalizer=Normalizer)
P1o = Propagator_Class(contour, f0=f0, Lx=Lx, Rx=Rx,
                       sign=-1, Normalizer=Normalizer)

In [ ]:
## plt.close('all')
P1e.plot_transform(part='real')
P1o.plot_transform(ax=plt.gca());

In [ ]:
contour = A.real_contour(6.2, 7, 150)

P2e = Propagator_Class(contour, f0=f0, Lx=Lx, Rx=Rx,
                       sign=1, Normalizer=Normalizer)
P2o = Propagator_Class(contour, f0=f0, Lx=Lx, Rx=Rx,
                       sign=-1, Normalizer=Normalizer)

In [ ]:
## plt.close('all')
P2e.plot_transform(part='real')
P2o.plot_transform(ax=plt.gca())

In [ ]:
contour = A.real_contour(7, A.Z_evanescent, 150)

P3e = Propagator_Class(contour, f0=f0, Lx=Lx, Rx=Rx,
                       sign=1, Normalizer=Normalizer)
P3o = Propagator_Class(contour, f0=f0, Lx=Lx, Rx=Rx,
                       sign=-1, Normalizer=Normalizer)

In [ ]:
## plt.close('all')
P3e.plot_transform(part='real')
P3o.plot_transform(ax=plt.gca())

In [ ]:
contour = A.real_contour(A.Z_evanescent, 25, 200)

P4e = Propagator_Class(contour, f0=f0, Lx=Lx, Rx=Rx,
                       sign=1, Normalizer=Normalizer)
P4o = Propagator_Class(contour, f0=f0, Lx=Lx, Rx=Rx,
                       sign=-1, Normalizer=Normalizer)

In [ ]:
## plt.close('all')
P4e.plot_transform(part='real')
P4o.plot_transform(ax=plt.gca())

### Combine propagators into one function

In [ ]:
Props = [P1e,
      P1o,
      P2e,
      P2o,
      P3e,
      P3o,
     ]

Evs = [P4e,
      P4o,
     ]

Prop = lambda x, z: sum([P.propagate(x, z) for P in Props])
Ev = lambda x, z: sum([P.propagate(x, z) for P in Evs])
Rad = lambda x, z: Prop(x,z) + Ev(x,z)
Full = Rad

diff = lambda x: Full(x, 0) - f0(x)

# 1D Plot


In [ ]:
xs = np.linspace(-6, 6, 1000)


In [ ]:
A.plot_field_1d(Full, xs=xs, label='estimate');
A.add_1d_plot(f0, xs=xs, label='$f_0$')

## Slice along Z axis

In [ ]:
A.plot_field_1d(Full, xs=0, zs=np.linspace(0, 50, 1000));


## 2D Plots

In [ ]:
xs = np.concatenate([np.linspace(-10,-1, 150), 
                     np.linspace(-1, 1, 50), 
                     np.linspace(1, 10, 150)])

In [ ]:
A.plot_field_2d(Full, xs=xs, zs=np.linspace(0, 10, 80), colorbar=False, levels=60);


In [ ]:
# A.plot_field_2d_surface(Full, xs=xs, colorbar=False, zoom=3.5,
#                         azim=-47, elev=30, 
#                         zmax=10, zref=150,
#                         rstride=1, cstride=1);

# Comparison to leaky mode representation

In [ ]:
# If using eigvec normalization, only need to do this for leaky field
# if not, need to perform circular contour integral around leaky-mode Z value.
coeff = quad(lambda x: FsZ(x) * f0(x), Lx, Rx, complex_func=True)[0]
Leaky = lambda x, z: coeff * FsZ(x, z)
Diff = lambda x, z: Leaky(x,z) - Full(x,z)

## Asymptotics of true and leaky mode in direction of propagation

In [ ]:
part='real'
A.plot_field_1d(Full, zs=np.linspace(0, 13, 1000), part=part, label='true');
A.add_1d_plot(Leaky, zs=np.linspace(0, 13, 1000), part=part, label='leaky');


### 2d Plots of real field and leaky mode field

In [ ]:
plt.close('all')
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, sharey=True, figsize=(11,8))

xs2 = np.linspace(-5, 5, 100)

zmin = 0
zmax = 25
zref = int((zmax-zmin)*20/2)
zs = np.linspace(zmin, zmax, zref)

In [ ]:
A.add_2d_plot(Full, ax=ax1, xs=xs2, zs=zs, plot_Rhos=True, colorbar=True);
ax1.set_title('True Field')

In [ ]:
A.add_2d_plot(Leaky, ax=ax2, xs=xs2, zs=zs, plot_Rhos=True, colorbar=True);
ax2.set_title('Leaky Mode Field')

In [ ]:
A.add_2d_plot(Diff, ax=ax3, xs=xs2, zs=zs, plot_Rhos=True, part='norm', colorbar=True);
ax3.set_title('Norm of Difference')

## Animations

### Transverse field at input

In [ ]:
A.animate_field_1d(Leaky, 'leaky', xs=xs, linewidth=1.5, secs=2.5, fps=26, color='C0')

In [ ]:
Video('leaky.mp4', html_attributes="controls loop")


#### Full field

In [ ]:
A.animate_field_2d(Full, 'true_field_2d_animation', xs=xs, zs=np.linspace(0,10,100), secs=2.5, fps=26)

In [ ]:
A.animate_field_2d(Leaky, 'leaky_mode_antiguide_contour',xs=xs, zs=np.linspace(0,10,100), secs=2.5, fps=26)

In [ ]:
Video('true_field_2d_animation.mp4', html_attributes="controls loop autoplay")


In [ ]:
Video('leaky_mode_antiguide_contour.mp4', html_attributes="controls loop autoplay")


#### Animated surface plot

In [ ]:
# Slow animation cell
# A.animate_field_2d_surface(Full, 'true_field_3d_animation', fps=20, secs=2,
#                            xs=np.linspace(-6,6,120), zmin=0, zmax=10, zref=160,
#                            rstride=1, cstride=1, part='real',
#                            azim=-47, elev=30, zoom=3.5, dpi=120)

In [ ]:
# Slow animation cell# Slow animation cell
A.animate_field_2d_surface(Leaky, 'leaky_field_3d_animation', fps=20, secs=2,
                           xs=np.linspace(-6,6,120), zmin=0, zmax=10, zref=160,
                           rstride=1, cstride=1, part='real',
                           azim=-47, elev=30, zoom=3.5, dpi=120)

In [ ]:
HTML("""
<video controls id="theVideo3"> <source src="true_field_3d_animation.mp4"> </video>
<video controls id="theVideo4"> <source src="leaky_field_3d_animation.mp4"> </video>
<script> video3 = document.getElementById("theVideo3")
video4 = document.getElementById("theVideo4")
video3.width = 800; video3.playbackRate = 1; video3.loop = true; video3.autoplay = true;
video4.width = 800; video4.playbackRate = 1; video4.loop = true; video4.autoplay = true;
</script> """)


## Adding steepest descent path contribution

In [ ]:
def sdp(x, guide=A):
    K = guide.K0 * guide.n0
    return -(K * x) / np.sqrt(K**2 + x**2)

In [ ]:
xmin, xmax= -8, .5
ymin, ymax = -1, 2
xs_sdp = np.linspace(xmin, xmax, 1000)
ys_sdp = sdp(xs_sdp)
msk = np.where((ys_sdp<ymax)*(ys_sdp>ymin))

In [ ]:
plotlogf(A.determinant_Z, xmin, xmax, ymin, ymax, 'TE', 'leaky', iref=200, rref=200, levels=100)
ax = plt.gca()
ax.plot(xs_sdp[msk], ys_sdp[msk]);
# ax.plot(xs[msk], -ys[msk]);

# Save data for visualization

In [ ]:
# propfuncs = [P1, P2, P3]
# evfuncs = [P4, P5, P6, P7, P8]

In [ ]:
# Prop = lambda x, z: sum([f.propagate(x, zs=z) for f in propfuncs])
# Ev = lambda x, z: sum([f.propagate(x, zs=z) for f in evfuncs]) 
# Full = lambda x, z: Prop(x, z) + Ev(x, z)

In [ ]:
xs = np.linspace(-15, 15, 1200)
zs = np.linspace(0, 20, 600)


In [ ]:
f0s = f0(xs)

In [ ]:
rad_ys_propagating = Prop(xs, zs)

In [ ]:
rad_ys_evanescent = Ev(xs, zs)


In [ ]:
Xs, Zs = np.meshgrid(xs, zs)

In [ ]:
def get_trig_indices(mesh_array):
    '''Get triangle index list for three js mesh formation via indexed buffer geometry.'''
    if len(mesh_array.shape) <= 1:
        raise ValueError('Provide meshgrid array as input.')
    Nz, Nx = mesh_array.shape[0], mesh_array.shape[1]
    l_inds = Nx * Nz
    indices =  np.arange(0, l_inds).reshape((Nx, Nz)).T  # Note transpose
    fbases = indices[:Nz-1, :Nx-1].T.flatten()  # Note transpose and index switch due to transpose
    f1 = np.array([[i, i+Nz, i+1] for i in fbases], dtype=int)
    f2 = np.array([[i+1, i+Nz, i+Nz+1] for i in fbases], dtype=int)
    return np.concatenate((f1, f2))


def get_base_vertices(Xs, Zs):
    '''Get vertices of triangles with zero height (y component in three.js).'''
    if Xs.shape != Zs.shape:
        raise ValueError('Provide meshgrid arrays (ie Xs, Zs = np.meshgrid(xs,zs)) as inputs.')
    Base = np.array([Xs, np.zeros_like(Xs), Zs], dtype=float)
    L = Xs.shape[0] * Xs.shape[1]
    base_vertices = Base.T.reshape((L,3))
    return base_vertices


def create_height_array_from_complex(ys, Xs):
    '''Get data for height from complex ys input (needs to be formatted as list:
        [ys[0].real, ys[0].imag, ys[1].real, ys[1].imag, ... ].'''
    L = Xs.shape[0] * Xs.shape[1]
    Ys = np.array([ys.real, ys.imag], dtype=float)
    heights = Ys.T.reshape((L,2))
    return heights


def save_trig_indices_as_json(filepath, mesh_array):
    indices = get_trig_indices(mesh_array)
    with open(filepath + '.json', 'w') as outfile:
        json.dump(indices.flatten().tolist(), outfile, separators=(',', ':'),
                  sort_keys=True, indent=4)


def save_base_vertices_as_json(filepath, Xs, Zs):
    base_vertices = get_base_vertices(Xs, Zs)
    with open(filepath + '.json', 'w') as outfile:
        json.dump(base_vertices.flatten().tolist(), outfile, separators=(',', ':'),
                  sort_keys=True, indent=4)
        

def save_heights_as_json(filepath, ys, Xs):
    heights= create_height_array_from_complex(ys, Xs)
    with open(filepath + '.json', 'w') as outfile:
        json.dump(heights.flatten().tolist(), outfile, separators=(',', ':'),
                  sort_keys=True, indent=4)

def save_values_as_json(filepath, f0s):
    with open(filepath + '.json', 'w') as outfile:
        json.dump(f0s.tolist(), outfile, separators=(',', ':'),
                  sort_keys=True, indent=4)




In [ ]:
prefix='/home/pv/local/slab/apps/antiguide/'

In [ ]:
save_trig_indices_as_json(prefix + 'trig_indices_new', Xs)


In [ ]:
save_base_vertices_as_json(prefix + 'base_vertices_new', Xs, Zs)


In [ ]:
save_heights_as_json(prefix + 'evanescent_heights', rad_ys_evanescent, Xs)


In [ ]:
save_heights_as_json(prefix + 'propagating_heights', rad_ys_propagating, Xs)


In [ ]:
save_values_as_json(prefix + 'f0s', f0s)


In [ ]:
save_values_as_json(prefix + 'x0s', xs)


# Discovery of conjugation issue!


### Our method

This still has issues: all the contour integrals go to zero.

In [ ]:
mode_type='leaky'

paper_method = False

In [ ]:
plotlogf(A.determinant_Z, - 1.4*A.K_low, -A.K_low, -1,1, field_type, mode_type,
         figsize=(10,5),iref=100, rref=100,levels=100)

hi kirbee ❤️

In [ ]:
Z_leaky = newton(A.determinant_Z, -6.5, args=('TE', mode_type))
Z_leaky, A.determinant_Z(Z_leaky, mode_type=mode_type)
plt.plot(Z_leaky.real, Z_leaky.imag, 'ro');

In [ ]:
circ_contour = {'center': Z_leaky, 'radius': 1e-4, 'N':50}

In [ ]:
P_leaky_even = Propagator_Class(f0=f0, Lx=-.5, Rx=.5, 
                     LZ=100, RZ=150, NZ=10, 
                     field_type=field_type, conjugate=False,
                     sign='+1', circ_contour=circ_contour, paper_method=paper_method)
P_leaky_odd = Propagator_Class(f0=f0, Lx=-.5, Rx=.5, 
                     LZ=100, RZ=150, NZ=10, 
                     field_type=field_type, conjugate=False,
                     sign='-1', circ_contour=circ_contour, paper_method=paper_method)

In [ ]:
plt.close('all')
fig, ax = plt.subplots(1, figsize=(8,4))
ax.plot(P_leaky_even.ts,  np.abs(P_leaky_even.alphas.imag));
ax.plot(P_leaky_odd.ts,  np.abs(P_leaky_odd.alphas));

In [ ]:
plt.close('all')
fig, ax = plt.subplots(1, figsize=(8,4))
ax.plot(P_leaky_even_theirs.ts,  np.abs(P_leaky_even_theirs.alphas.real));
ax.plot(P_leaky_odd_theirs.ts,  np.abs(P_leaky_odd_theirs.alphas));

In [ ]:
P_ls = [P_leaky_even, P_leaky_odd]

full_leaky = lambda x : sum([P.slice_propagate(x) for P in P_ls])
Full_leaky = lambda x, z : sum([P.propagate(x, zs=z) for P in P_ls])

In [ ]:
A.plot_field_1d(full_leaky, xs=xs, part='real', color='blue');

#### Asymptotics of true and leaky mode (found with contour integral)



In [ ]:
slice_at_x = 0
constant_variable = 'x'

full_z_leaky_ours = lambda z: sum([P.slice_propagate(z, slice_at=slice_at_x, constant_variable=constant_variable) for P in P_ls])
full_z = lambda z: sum([P.slice_propagate(z, slice_at=slice_at_x, constant_variable=constant_variable) for P in Ps])



In [ ]:
part = 'real'
A.plot_field_1d(full_z, xs=np.linspace(0, 13, 1000), part=part, plot_Rhos=False, linewidth=2,label='true');
A.add_1d_plot(full_z_leaky_ours, xs=np.linspace(0, 13, 1000), part=part, label='leaky ours');
A.add_1d_plot(full_z_leaky_theirs, xs=np.linspace(0, 13, 1000), part=part, linestyle='--', label='leaky theirs');


#### Issues:

- Using our method, even though there appears to be a pole in the integrand where we need it to be, when we do a circular contour integral it appears to zero out.  Everything should be right, but we get zero...  This happens for all other pole locations too: -Z, Z.conj(), -Z.conj().

- This is strange because when we do this for the delta function input everything works out: we get convergent leaky mode fields.


#### Seem to have found issue: We need to use non-conjugated version of expansion and orthogonality



## Adding steepest descent path contribution

In [ ]:
def sdp(x, guide=A):
    K = guide.K0 * guide.n0
    return -(K * x) / np.sqrt(K**2 + x**2)

In [ ]:
xmin, xmax= -8, .5
ymin, ymax = -1, 2
xs_sdp = np.linspace(xmin, xmax, 1000)
ys_sdp = sdp(xs_sdp)
msk = np.where((ys_sdp<ymax)*(ys_sdp>ymin))

In [ ]:
plotlogf(A.determinant_Z, xmin, xmax, ymin, ymax, 'TE', 'leaky', iref=200, rref=200, levels=100)
ax = plt.gca()
ax.plot(xs_sdp[msk], ys_sdp[msk]);
# ax.plot(xs[msk], -ys[msk]);